In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
    

In [15]:
X, y = make_classification(n_samples=100, n_features=4, n_classes=2, random_state=42)
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(4)])
df['target'] = y

In [16]:
weights = np.zeros(X.shape[1])
intercept = 0.0

In [17]:
def predict_proba(X):
    y_pred = X @ weights + intercept
    return 1 / (1 + np.exp(-y_pred))

In [18]:
def predict(X, threshold=0.5):
    proba = predict_proba(X)
    return (proba >= threshold).astype(int)

In [19]:
def intercept_dv(y_pred, y_true):
    return np.mean(y_pred - y_true)

In [20]:
def weights_dv(y_pred, y_true, X):
    return (X.T @ (y_pred - y_true)) / len(y_true)

In [21]:
def train(X, y, lr = 0.01, max_itters=100):
    global weights, intercept
    for _ in range(max_itters):
        y_pred = predict_proba(X)
        w_dv = weights_dv(y_pred, y, X)
        b_dv = intercept_dv(y_pred, y)
        
        weights -= lr * w_dv
        intercept -= lr * b_dv

In [22]:
train(X, y)
y_pred = predict(X)
y_proba = predict_proba(X)


df_preds = pd.DataFrame({"probability": y_proba, "predicted_class": y_pred, "true_class": y})
print(df_preds, "\nAccuracy:", accuracy_score(y, y_pred))

    probability  predicted_class  true_class
0      0.633570                1           1
1      0.455613                0           0
2      0.759793                1           1
3      0.167165                0           0
4      0.178676                0           0
..          ...              ...         ...
95     0.184660                0           0
96     0.564980                1           1
97     0.397848                0           0
98     0.796701                1           1
99     0.780024                1           1

[100 rows x 3 columns] 
Accuracy: 0.93


In [ ]:
from simpleml.linear_models import SGDClassifier
from simpleml.pipeline import make_pipeline
from simpleml.preprocessing import StandardScaler

X, y = make_classification(n_samples=100, n_features=4, n_classes=2, random_state=42)
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(4)])
df['target'] = y

pipe = SGDClassifier(random_state=42, lr = 0.01)

pipe.fit(X, y)

y_pred = pipe.predict(X)
y_proba = pipe.predict_proba(X)

df_preds = pd.DataFrame({"probability": y_proba, "predicted_class": y_pred, "true_class": y})
print(y_pred)
print(df_preds, "\nAccuracy:", accuracy_score(y, y_pred))

Val Loss: 0.5427655014217257
[1 0 1 0 0 1 0 1 0 0 1 0 1 0 0 1 1 1 1 1 1 0 0 0 0 1 1 0 0 0 1 1 1 0 1 1 1
 0 1 1 1 0 0 0 0 1 1 1 1 0 1 0 0 0 0 1 1 0 0 1 0 1 1 1 0 0 0 0 0 1 0 1 0 1
 1 0 0 1 1 0 1 1 0 0 0 1 1 0 1 1 1 1 0 0 0 0 1 0 1 1]
    probability  predicted_class  true_class
0      0.711613                1           1
1      0.558481                0           0
2      0.756982                1           1
3      0.423443                0           0
4      0.431163                0           0
..          ...              ...         ...
95     0.497482                0           0
96     0.639493                1           1
97     0.582154                0           0
98     0.736358                1           1
99     0.763431                1           1

[100 rows x 3 columns] 
Accuracy: 0.93


In [24]:


model = LogisticRegression()
model.fit(X, y)

y_pred = model.predict(X)
y_proba = model.predict_proba(X)[:, 1]

df_preds = pd.DataFrame({"probability": y_proba, "predicted_class": y_pred, "true_class": y})
print(df_preds, "\nAccuracy:", accuracy_score(y, y_pred))

    probability  predicted_class  true_class
0      0.986266                1           1
1      0.057436                0           0
2      0.998721                1           1
3      0.000233                0           0
4      0.000323                0           0
..          ...              ...         ...
95     0.005728                0           0
96     0.682355                1           1
97     0.154687                0           0
98     0.995640                1           1
99     0.999105                1           1

[100 rows x 3 columns] 
Accuracy: 0.99


In [25]:
X = np.array([0.5, 1.0, 0.5, 1.0])

m = (2 / len(X)) * sum(X)

m1 = 2 * np.mean(X)

print("m:", m)
print("m1:", m1)

m: 1.5
m1: 1.5


## **Okay so now that I got an Idea of how logistic regression works with logloss and sigmoid**
## The next step is making it into an SGD Classifier that uses
### - Stable logloss instead of normal logloss to prevent infinites and NANs
####  l = ln(1 + e^z) - yz or logaddexp(0, z) - y*z
### - Stable Sigmoid instead of normal sigmoid also to prevent explosions and NANs
####  if z >= 0  sigmoid(z) = 1 / (1 + e^-z)
####  if z < 0 sigmoid(z) = e^z/ (1 + e^z)
### - l2 regurlarization same as the one in SGD Regressor just usig logloss instead of mse
### - lr scheduler similar to the SGD Regressor
### - early stopping using logloss on validation
### - adding class_weights hyperparameter for imbalanced labels & decision_function() for debugging & ROC
### - Testing and final modifications before adding anything new